# 06 - QLoRA fine-tune Gemma-2-9B-it as the meta-model

Trains on `artifacts/meta_jsonl/train.jsonl`, evaluates on `val.jsonl`, saves the LoRA adapter to `artifacts/lora_adapter/`.

Requirements: a CUDA GPU with ~24 GB VRAM (e.g. A100 40GB or A6000) and an `HF_TOKEN` with Gemma-2 license accepted on HuggingFace.

In [ ]:
%pip install -q 'transformers>=4.44' 'peft>=0.11' 'trl>=0.9' 'bitsandbytes>=0.43' 'accelerate>=0.33' datasets sentencepiece

In [ ]:
import os, sys, json

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling'
TMP_ROOT = '/content/ensemble_tmp'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    sys.path.insert(0, str(LOCAL.parent))

from tm_research.ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

from tm_research.ensemble.utils_io import (
    META_JSONL_DIR, LORA_DIR, ARTIFACTS_DIR, load_label_map
)
label_map = load_label_map()
print('classes', label_map.class_names)

In [ ]:
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError(
        'Set HF_TOKEN environment variable with a token that has accepted the '
        'Gemma-2 license at https://huggingface.co/google/gemma-2-9b-it. '
        'On Colab: open the key icon in the sidebar, add an HF_TOKEN secret, '
        'grant this notebook access, then re-run setup_colab().'
    )

MODEL_NAME = 'google/gemma-2-9b-it'
MAX_SEQ_LEN = 1024
TRAIN_WORK_DIR = str(paths.lora_work / 'gemma2_9b')
FINAL_ADAPTER_DIR = str(LORA_DIR)
print('train work dir (ephemeral):', TRAIN_WORK_DIR)
print('final adapter dir (Drive):', FINAL_ADAPTER_DIR)

## Build chat-formatted dataset

Each example becomes a Gemma chat with three turns: `system` (task description), `user` (the structured-token block from step 5), and `assistant` (the `<label>...</label>` completion). We use `tokenizer.apply_chat_template` and let `SFTTrainer` mask everything but the assistant span via the `completion_only` collator.

In [ ]:
from datasets import load_dataset

raw = load_dataset(
    'json',
    data_files={
        'train': str(META_JSONL_DIR / 'train.jsonl'),
        'validation': str(META_JSONL_DIR / 'val.jsonl'),
    },
)
raw

In [ ]:
CLASS_LIST = ', '.join(label_map.class_names)
SYSTEM_PROMPT = (
    'You are an emotion classifier for Vietnamese social-media text. '
    f'Pick exactly one label from: {CLASS_LIST}. '
    'You receive the input text and the per-class probability distributions of five base models, '
    'each with its weight (higher = more reliable on validation). '
    'You also receive their weighted average. '
    'Output ONLY the final answer in the exact format: <label>LABEL</label>.'
)

def to_messages(example):
    messages = [
        {'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + example['prompt']},
        {'role': 'assistant', 'content': example['completion']},
    ]
    return {'messages': messages}

ds = raw.map(to_messages, remove_columns=raw['train'].column_names)
print(ds)
print('first messages:', ds['train'][0]['messages'])

## Load model in 4-bit + attach LoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.environ['HF_TOKEN'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ATTN_IMPL = os.environ.get('TM_ATTN_IMPL', 'sdpa')

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation=ATTN_IMPL,
    token=os.environ['HF_TOKEN'],
)
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## SFTTrainer (assistant-only loss)

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

_sft_config_params = inspect.signature(SFTConfig.__init__).parameters
_sft_kwargs = dict(
    output_dir=TRAIN_WORK_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    weight_decay=0.0,
    bf16=True,
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    packing=False,
    completion_only_loss=True,
    report_to='none',
    seed=42,
)
if 'max_length' in _sft_config_params:
    _sft_kwargs['max_length'] = MAX_SEQ_LEN
elif 'max_seq_length' in _sft_config_params:
    _sft_kwargs['max_seq_length'] = MAX_SEQ_LEN
sft_args = SFTConfig(**_sft_kwargs)

_sft_trainer_params = inspect.signature(SFTTrainer.__init__).parameters
_trainer_kwargs = dict(
    model=model,
    args=sft_args,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
)
if 'processing_class' in _sft_trainer_params:
    _trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in _sft_trainer_params:
    _trainer_kwargs['tokenizer'] = tokenizer

trainer = SFTTrainer(**_trainer_kwargs)
trainer.train()

In [ ]:
os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)
trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
with open(ARTIFACTS_DIR / 'lora_train_meta.json', 'w', encoding='utf-8') as f:
    json.dump({
        'base_model': MODEL_NAME,
        'final_adapter_dir': FINAL_ADAPTER_DIR,
        'train_work_dir': TRAIN_WORK_DIR,
        'system_prompt': SYSTEM_PROMPT,
        'max_seq_len': MAX_SEQ_LEN,
    }, f, ensure_ascii=False, indent=2)
print('LoRA adapter saved to', FINAL_ADAPTER_DIR, '(persistent, on Drive)')
print('Intermediate trainer checkpoints are in', TRAIN_WORK_DIR, '(ephemeral, /content)')

In [ ]:
from tm_research.ensemble.utils_io import push_artifacts_to_persistent
push_artifacts_to_persistent()